In [ ]:
# https://towardsdev.com/building-visionrag-ai-powered-image-search-with-llama-3-2-qdrant-and-litserve-10bf22df5d41
# https://medium.com/kx-systems/guide-to-multimodal-rag-for-images-and-text-10dab36e3117

In [ ]:
import os
import base64
import requests
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from sentence_transformers import SentenceTransformer
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import uuid

In [ ]:
# Configuration for Local Ollama
OLLAMA_SERVER_URL = "http://localhost:11434"

# Ensure these correspond to the exact model names you have pulled in Ollama
OLLAMA_VISION_MODEL = "deepseek-ocr" # Replace with actual deepseek vision/ocr model name in your ollama setup
OLLAMA_REASONING_MODEL = "deepseek-r1" # Replace with actual deepseek-r1 model name in your ollama setup


In [ ]:
# ========= 1. Ollama DeepSeek OCR =========
def deepseek_ocr(image_path):
    endpoint = f"{OLLAMA_SERVER_URL}/api/generate"

    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode("utf-8")

    payload = {
        "model": OLLAMA_VISION_MODEL,
        "prompt": "Tell me the brand of this battery. Just give me the brand name. Do not give other explanation and text. If you cannot recognize, Just tell us NA:",
        "images": [img_b64],
        "stream": False
    }
    
    try:
        resp = requests.post(endpoint, json=payload)
        resp.raise_for_status()
        return resp.json()["response"].strip()
    except Exception as e:
        print(f"Error calling Ollama OCR: {e}")
        return "NA"


In [ ]:
# ========= 2. Embedding =========
device = "cuda" if torch.cuda.is_available() else "cpu"
# For Mac, you might want "mps"; for AMD "cuda" works via ROCm if installed correctly, else "cpu"

# Text embedding (GTE-base)
text_model = SentenceTransformer("thenlper/gte-base", device=device)

In [ ]:
# Image embedding (CLIP)
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
def embed_text(text):
    return text_model.encode(text, convert_to_numpy=True)

def embed_image(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        emb = clip_model.get_image_features(**inputs)
    return emb.squeeze().cpu().numpy()

In [ ]:
# ========= 3. Qdrant =========
client = QdrantClient(path="/qdrant_data")  

In [ ]:
# Install collection, including text_embedding + image_embedding
client.recreate_collection(
    collection_name="multimodal_docs",
    vectors_config={
        "text_embedding": VectorParams(size=768, distance=Distance.COSINE),
        "image_embedding": VectorParams(size=512, distance=Distance.COSINE),
    },
)

In [ ]:
# ========= 4. Put images into ocr, text embedding, image embedding =========
def insert_image_doc(image_path):
    ocr_text = deepseek_ocr(image_path)
    text_vec = embed_text(ocr_text)
    img_vec = embed_image(image_path)

    point_id = str(uuid.uuid4())
    client.upsert(
        collection_name="multimodal_docs",
        points=[
            PointStruct(
                id=point_id,
                vector={
                    "text_embedding": text_vec.tolist(),
                    "image_embedding": img_vec.tolist(),
                },
                payload={
                    "source": image_path,
                    "ocr_text": ocr_text,
                },
            )
        ],
    )
    print(f"Inserted {image_path} with OCR: {ocr_text[:50]}...")

In [ ]:
# ========= 5. Query =========
def search_by_text(query, top_k=3):
    q_vec = embed_text(query)
    results = client.search(
        collection_name="multimodal_docs",
        query_vector=("text_embedding", q_vec.tolist()),
        limit=top_k,
    )
    return results

def search_by_image(image_path, top_k=3):
    q_vec = embed_image(image_path)
    results = client.search(
        collection_name="multimodal_docs",
        query_vector=("image_embedding", q_vec.tolist()),
        limit=top_k,
    )
    return results

def search_hybrid(query_text=None, query_image=None, alpha=0.7, top_k=3):
    # Optional naive hybrid implementation
    pass

In [ ]:
from glob import glob

# ========= 6. Read all images in the folder =========
def insert_folder_images(folder_path):
    image_paths = glob(os.path.join(folder_path, "*.png")) +                   glob(os.path.join(folder_path, "*.jpg")) +                   glob(os.path.join(folder_path, "*.jpeg"))

    for img_path in image_paths:
        try:
            insert_image_doc(img_path)
        except Exception as e:
            print(f"❌ Failed {img_path}: {e}")

In [ ]:
# ========= 7. Retrieval =========
def retrieve_contexts(query_text=None, query_image=None, mode="text", top_k=3):
    if mode == "text":
        return search_by_text(query_text, top_k)
    elif mode == "image":
        return search_by_image(query_image, top_k)
    elif mode == "hybrid":
        return search_hybrid(query_text=query_text, query_image=query_image, alpha=0.7, top_k=top_k)
    else:
        raise ValueError("mode must be 'text', 'image', or 'hybrid'")

In [ ]:
# ========= 8. DeepSeek R1 Generation =========
def generate_rag_answer(query_text, contexts):
    endpoint = f"{OLLAMA_SERVER_URL}/api/generate"
    
    # Format the retrieved context into a single string
    context_str = "\n".join([f"- Found Brand: {hit.payload.get('ocr_text', 'Unknown')} (Source: {hit.payload.get('source', '')})" for hit in contexts])
    
    prompt = f"""You are a helpful AI assistant analyzing battery brands.
Context Information:
{context_str}

User Question: {query_text}

Provide a concise answer based ONLY on the context information provided above.
"""

    payload = {
        "model": OLLAMA_REASONING_MODEL,
        "prompt": prompt,
        "stream": False
    }
    
    try:
        resp = requests.post(endpoint, json=payload)
        resp.raise_for_status()
        return resp.json()["response"].strip()
    except Exception as e:
        print(f"Error calling Ollama DeepSeek R1: {e}")
        return "Failed to generate answer."


In [ ]:
# ========= 9. Full RAG Pipeline =========
def rag_pipeline(query_text, mode="text", top_k=3):
    print(f"Query: {query_text}")
    print("Retrieving contexts...")
    hits = retrieve_contexts(query_text=query_text, mode=mode, top_k=top_k)
    
    print("Retrieved:")
    for h in hits:
        print(f"  - {h.payload}")
        
    print("\nGenerating answer with DeepSeek R1...")
    answer = generate_rag_answer(query_text, hits)
    print("====================================")
    print(f"Answer:\n{answer}")
    return answer

In [ ]:
if __name__ == "__main__":
    folder = "image/rag"  # Image folder path
    insert_folder_images(folder)

In [ ]:
print("\n🔍 Testing Text Query RAG Pipeline:")
_ = rag_pipeline(query_text="What panasonic batteries do we have?", mode="text", top_k=3)